In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.svm import SVR

In [11]:
df = pd.read_csv('laptop_data.csv')
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

#  Define Modern Market Data (Apple Silicon, 14th Gen Intel, RTX 40 Series, etc.)
apple_specs = [
    ("Apple", "Ultrabook", 13.6, "IPS Panel Liquid Retina Display 2560x1664", "Apple M2", "8GB", "256GB SSD", "Apple M2 8-Core GPU", "macOS", "1.24kg", 99900),
    ("Apple", "Ultrabook", 13.6, "IPS Panel Liquid Retina Display 2560x1664", "Apple M2", "16GB", "512GB SSD", "Apple M2 10-Core GPU", "macOS", "1.24kg", 139900),
    ("Apple", "Ultrabook", 15.3, "IPS Panel Liquid Retina Display 2880x1864", "Apple M3", "16GB", "512GB SSD", "Apple M3 10-Core GPU", "macOS", "1.51kg", 154900),
    ("Apple", "Ultrabook", 14.2, "IPS Panel Liquid Retina XDR Display 3024x1964", "Apple M3 Pro", "18GB", "512GB SSD", "Apple M3 Pro 14-Core GPU", "macOS", "1.61kg", 199900),
    ("Apple", "Ultrabook", 16.2, "IPS Panel Liquid Retina XDR Display 3456x2234", "Apple M3 Max", "36GB", "1TB SSD", "Apple M3 Max 30-Core GPU", "macOS", "2.14kg", 319900)
]

gaming_specs = [
    ("Asus", "Gaming", 15.6, "Full HD 1920x1080", "Intel Core i7 13700H", "16GB", "1TB SSD", "Nvidia GeForce RTX 4060", "Windows 11", "2.20kg", 135000),
    ("Lenovo", "Gaming", 16.0, "IPS Panel 2560x1600", "AMD Ryzen 7 7840HS", "16GB", "512GB SSD", "Nvidia GeForce RTX 4050", "Windows 11", "2.40kg", 105000),
    ("Dell", "Gaming", 15.6, "Full HD 1920x1080", "Intel Core i9 13900HX", "32GB", "1TB SSD", "Nvidia GeForce RTX 4080", "Windows 11", "2.81kg", 285000),
    ("Asus", "Gaming", 18.0, "IPS Panel 2560x1600", "Intel Core i9 14900HX", "64GB", "2TB SSD", "Nvidia GeForce RTX 4090", "Windows 11", "3.10kg", 450000)
]

ultrabook_specs = [
    ("Dell", "Ultrabook", 13.4, "OLED Touchscreen 2880x1800", "Intel Core Ultra 7 155H", "16GB", "512GB SSD", "Intel Arc Graphics", "Windows 11", "1.19kg", 155000),
    ("Lenovo", "Ultrabook", 14.0, "OLED 2880x1800", "AMD Ryzen 7 8840HS", "16GB", "1TB SSD", "AMD Radeon 780M", "Windows 11", "1.35kg", 105000),
    ("Asus", "Ultrabook", 14.0, "OLED Touchscreen 2880x1800", "Intel Core Ultra 9 185H", "32GB", "1TB SSD", "Intel Arc Graphics", "Windows 11", "1.20kg", 180000)
]

#  Combine and weigh the modern data heavily so the model prioritizes it
modern_df = pd.DataFrame(apple_specs + gaming_specs + ultrabook_specs, columns=df.columns)
modern_df = pd.concat([modern_df] * 12, ignore_index=True) # Multiply to give it presence
modern_df['Price'] = modern_df['Price'] * np.random.uniform(0.95, 1.05, size=len(modern_df)) # Add slight price variance

df = pd.concat([df, modern_df], ignore_index=True)
print(f"Dataset successfully loaded and updated! Shape: {df.shape}")

Dataset successfully loaded and updated! Shape: (1687, 11)


In [12]:
df['Ram'] = df['Ram'].str.replace('GB','')
df['Weight'] = df['Weight'].str.replace('kg','')

df['Ram'] = df['Ram'].astype('int32')
df['Weight'] = df['Weight'].astype('float32')

df.head()

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,47895.5232
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,30636.0000
3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,135195.3360
4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080


In [13]:
df['Touchscreen'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)
df['Ips'] = df['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)

# Extract Resolution and calculate Pixels Per Inch (PPI)
new = df['ScreenResolution'].str.split('x', n=1, expand=True)
df['X_res'] = new[0].str.replace(',','').str.findall(r'(\d+\.?\d+)').apply(lambda x: x[0]).astype('int')
df['Y_res'] = new[1].astype('int')

df['ppi'] = (((df['X_res']**2) + (df['Y_res']**2))**0.5 / df['Inches']).astype('float')

df.drop(columns=['ScreenResolution', 'Inches', 'X_res', 'Y_res'], inplace=True)
df.head()

,Company,TypeName,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price,Touchscreen,Ips,ppi
0,Apple,Ultrabook,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832,0,1,226.983005
1,Apple,Ultrabook,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,47895.5232,0,0,127.677940
2,HP,Notebook,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,30636.0000,0,0,141.211998
3,Apple,Ultrabook,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,135195.3360,0,1,220.534624
4,Apple,Ultrabook,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080,0,1,226.983005


In [14]:
df['Cpu Name'] = df['Cpu'].apply(lambda x: " ".join(x.split()[0:3]))

def fetch_processor(text):
    if text in ['Intel Core i7', 'Intel Core i5', 'Intel Core i3']:
        return text
    elif text.split()[0] == 'Intel':
        return 'Other Intel Processor'
    else:
        return 'AMD Processor' # Groups AMD and Apple Silicon into non-Intel

df['Cpu brand'] = df['Cpu Name'].apply(fetch_processor)
df.drop(columns=['Cpu', 'Cpu Name'], inplace=True)

In [15]:
df['Memory'] = df['Memory'].astype(str).replace(r'\.0', '', regex=True)
df['Memory'] = df['Memory'].str.replace('GB', '').str.replace('TB', '000')

new = df['Memory'].str.split("+", n=1, expand=True)
df1 = new[0].str.strip()
df2 = new[1].fillna("0")

df1_HDD = df1.apply(lambda x: 1 if "HDD" in x else 0)
df1_SSD = df1.apply(lambda x: 1 if "SSD" in x else 0)
df1 = df1.str.replace(r'\D', '', regex=True).astype(int)

df2_HDD = df2.apply(lambda x: 1 if "HDD" in x else 0)
df2_SSD = df2.apply(lambda x: 1 if "SSD" in x else 0)
df2 = df2.str.replace(r'\D', '', regex=True).astype(int)

df['HDD'] = (df1 * df1_HDD) + (df2 * df2_HDD)
df['SSD'] = (df1 * df1_SSD) + (df2 * df2_SSD)

df.drop(columns=['Memory'], inplace=True)

In [16]:
df['Gpu brand'] = df['Gpu'].apply(lambda x: x.split()[0])
df = df[df['Gpu brand'] != 'ARM'] # Remove extreme outliers if any
df.drop(columns=['Gpu'], inplace=True)

def cat_os(inp):
    if inp in ['Windows 10', 'Windows 7', 'Windows 10 S', 'Windows 11']:
        return 'Windows'
    elif inp in ['macOS', 'Mac OS X']:
        return 'Mac'
    else:
        return 'Others/No OS/Linux'

df['os'] = df['OpSys'].apply(cat_os)
df.drop(columns=['OpSys'], inplace=True)

In [17]:
# 1. Setup Train/Test Split
X = df.drop(columns=['Price'])
y = np.log(df['Price'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# 2. Setup the Column Transformer for Categorical Data
step1 = ColumnTransformer(transformers=[
    ('col_tnf', OneHotEncoder(sparse_output=False, drop='first'), [0,1,7,10,11])
], remainder='passthrough')

# 3. Create a dictionary of models to test
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=10),
    'Lasso': Lasso(alpha=0.001),
    'K-Neighbors': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(max_depth=8),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_samples=0.5, max_features=0.75, max_depth=15, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_features=0.5),
    'AdaBoost': AdaBoostRegressor(n_estimators=15, learning_rate=1.0),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, max_features=0.75, max_depth=15, n_jobs=-1),
    'Support Vector Regression': SVR(kernel='rbf', C=10000, epsilon=0.1)
}

# 4. Loop through the models, train, and evaluate
model_results = []

best_r2_score = -float('inf')
best_model_name = ""
best_model = None

for name, model in models.items():
    # Build pipeline
    pipe = Pipeline([
        ('step1', step1),
        ('step2', model)
    ])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Predict
    y_pred = pipe.predict(X_test)
    
    # Evaluate
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Save results
    model_results.append({
        'Model Name': name,
        'R2 Score': r2,
        'Mean Absolute Error': mae
    })
    if r2 > best_r2_score:
        best_r2_score = r2
        best_model_name = name
        best_model = model

# 5. Display the leaderboard!
results_df = pd.DataFrame(model_results).sort_values(by='R2 Score', ascending=False)
print("🏆 Model Leaderboard 🏆")
results_df.head(10)

🏆 Model Leaderboard 🏆


,Model Name,R2 Score,Mean Absolute Error
5,Random Forest,0.936835,0.140884
6,Gradient Boosting,0.930061,0.158391
8,Extra Trees,0.924697,0.146301
9,Support Vector Regression,0.919954,0.168959
4,Decision Tree,0.914912,0.169536
3,K-Neighbors,0.898881,0.176229
0,Linear Regression,0.875765,0.209277
2,Lasso,0.868703,0.220059
1,Ridge,0.867767,0.222778
7,AdaBoost,0.856436,0.255076


In [18]:
final_pipe = Pipeline([
    ('step1', step1),
    ('step2', best_model)
])

final_pipe.fit(X, y)

# Export
pickle.dump(df, open('df.pkl', 'wb'))
pickle.dump(final_pipe, open('pipe.pkl', 'wb'))
print(f"🎉 Export complete! The winning model has been saved as 'pipe.pkl'.")

🎉 Export complete! The winning model has been saved as 'pipe.pkl'.
